# Diagnostic — a meaningful-change reference for detector #1 (colab_11 v2)

colab_11 v2's detector #1 came back with a held-out cosine drift of 0.0051. The companion magnitude diagnostic confirmed this is a real, small rotation of the pooled embedding (about 5.8 degrees), clearly above the near-zero noise floor but of unknown size relative to a *biologically meaningful* difference. A cosine number in isolation cannot say whether 0.0051 is a substantial reorganization or a negligible nudge -- that judgement needs a reference contrast measured in the same units.

This notebook builds that reference by measuring how far apart genuinely distinct cell states already sit in the **frozen zero-shot Geneformer embedding** -- the same space detector #1's drift is measured in -- so 0.0051 can be read as a fraction of a known biological difference. Three landmarks are computed, all in cosine-distance units:

- **Lineage (astrocyte vs microglia)** -- the largest unambiguous biological axis; a clean upper anchor.
- **Substate, pooled** -- homeostatic vs activated (microglia) and resting vs reactive (astrocyte), computed across all cells. These substate labels are known to be donor/study-confounded, so this number is inflated by batch structure and is reported as a diagnostic, not a trusted reference.
- **Substate, within-donor** -- the same substate contrast computed *inside each donor* and then medianed across donors. Measuring the gap inside one donor removes the donor/study inflation, giving a clean estimate of the biological substate difference at the scale continued pretraining could plausibly move. This is the primary meaningful-change reference.

The notebook is standalone and read-only: it loads one already-saved embedding from Drive and computes distances on it. It re-runs no training or embedding step and needs no GPU.

### Setup — install `anndata`

Colab's base runtime ships `numpy`/`pandas` but not `anndata`, which is needed to read the saved `.h5ad` embedding. `anndata` is pinned to `0.13.2` -- the version used by the companion magnitude diagnostic that read the same file -- rather than left unpinned: on a kernel where `numpy` had already been imported, an unpinned install pulled a newer release that upgraded `numpy` on disk and broke the import with a load-time version skew. Pinning keeps the read reproducible. The resolved version prints below for the software-version record.

**Operational note:** even with the pin, this install crashed the 1a cell (kernel death, not a Python `ImportError`) on first run -- a `Runtime > Restart session` after this cell, before running 1a, was required to get a clean import. Restart here as a matter of course rather than only after a crash.

In [6]:
!pip install -q "anndata==0.13.2"
from importlib.metadata import version
print("anndata", version("anndata"))

anndata 0.13.2


> **Interpretation — setup.** The printed `anndata 0.13.2` confirms the pinned install resolved to the intended version, the same version the companion magnitude diagnostic used to read this same file -- so every distance computed below is measured with an identical read path to that companion result. As noted above, this cell still crashed the kernel on first attempt despite the pin; the numbers below come from the post-restart run.

### 1a — load the frozen zero-shot embedding and inspect its cell-state composition

Load the frozen zero-shot baseline (`glia_geneformer_zeroshot.h5ad`, colab_09) -- the reference is a property of this embedding's internal geometry alone, so no post-CPT file is needed. The `lineage` and `substate` labels carried on the object are what define the reference contrasts, so before computing any distance, print the cell counts of each lineage and the lineage-by-substate cross-tabulation. This confirms the label categories are the expected ones (microglia: homeostatic / activated / intermediate; astrocyte: resting / reactive / intermediate) and exposes the group sizes -- centroids computed from thin groups are unstable, so the sizes need to be seen before the numbers built on them are trusted. The per-donor breakdown of each substate pole is also printed, since the within-donor reference (cell 4a) depends on donors carrying enough of both poles.

In [7]:
import os
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
ZEROSHOT_PATH = os.path.join(DRIVE_ROOT, "geneformer", "glia_geneformer_zeroshot.h5ad")
assert os.path.exists(ZEROSHOT_PATH), f"missing saved embedding: {ZEROSHOT_PATH}"

zs = ad.read_h5ad(ZEROSHOT_PATH)
print("zero-shot:", zs.shape)

for col in ("lineage", "substate", "donor_id"):
    assert col in zs.obs.columns, f"missing obs column: {col}"

# X = non-normalized mean-pooled 768-dim embedding, as detector #1 uses
X = zs.X.toarray() if sp.issparse(zs.X) else np.asarray(zs.X)
X = np.asarray(X, dtype=np.float32)
assert X.ndim == 2 and X.shape[1] == 768, f"expected dense 768-d embedding, got shape {X.shape}"
assert np.isfinite(X).all(), "embedding contains non-finite values"
lineage = zs.obs["lineage"].astype(str).values
substate = zs.obs["substate"].astype(str).values
donor = zs.obs["donor_id"].astype(str).values

print("\nlineage counts:")
print(pd.Series(lineage).value_counts())
print("\nlineage x substate:")
print(pd.crosstab(pd.Series(lineage, name="lineage"), pd.Series(substate, name="substate")))

# substate poles contrasted per lineage (each lineage's own intermediate bucket excluded -- contrast only the two extreme poles)
POLES = {"microglia": ("homeostatic", "activated"), "astrocyte": ("resting", "reactive")}
# fail loud if any expected substate label string is absent for a lineage (silent empty poles -> NaN centroids)
for lin, (p1, p2) in POLES.items():
    for pole in (p1, p2):
        n = int(((lineage == lin) & (substate == pole)).sum())
        assert n > 0, f"no cells for {lin!r} substate {pole!r} -- POLES strings do not match the object's labels"

print("\nper-donor cell counts at each substate pole (donors carrying >=1 of a pole):")
for lin, (p1, p2) in POLES.items():
    for pole in (p1, p2):
        m = (lineage == lin) & (substate == pole)
        n_don = pd.Series(donor[m]).nunique()
        print(f"  {lin:10s} {pole:12s} | {int(m.sum()):6d} cells across {n_don} donors")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
zero-shot: (142588, 768)

lineage counts:
astrocyte    87783
microglia    54805
Name: count, dtype: int64

lineage x substate:
substate   activated  homeostatic  intermediate  reactive  resting
lineage                                                           
astrocyte          0            0         11171     28465    48147
microglia      12109        25845         16851         0        0

per-donor cell counts at each substate pole (donors carrying >=1 of a pole):
  microglia  homeostatic  |  25845 cells across 142 donors
  microglia  activated    |  12109 cells across 144 donors
  astrocyte  resting      |  48147 cells across 145 donors
  astrocyte  reactive     |  28465 cells across 145 donors


> **Interpretation — 1a.** The loaded object is 142,588 cells x 768 dims, splitting into 87,783 astrocyte / 54,805 microglia -- an exact match to the documented FM substrate, confirming this is the right frozen file. The lineage x substate crosstab shows the expected microglia categories (homeostatic 25,845 / activated 12,109 / intermediate 16,851) and confirms something not previously on record: astrocyte carries a third substate label too -- `intermediate` (11,171 cells), alongside resting (48,147) and reactive (28,465) -- rather than being a strict binary resting/reactive split. This mirrors microglia's own three-way scheme and is handled correctly here: `POLES` only contrasts the two extreme poles per lineage, so astro's intermediate bucket is excluded from every downstream distance exactly as microglia's is. Combined, 28,022 cells (16,851 + 11,171, about 19.7% of the object) are excluded from every distance computed in this notebook -- worth stating explicitly here since no cell prints that drop tally on its own; it is only recoverable from this crosstab. The per-donor counts printed here (142-145 donors carrying >=1 cell of a given pole, out of ~145 total) only check presence at the >=1-cell level -- they are not the >=50-cells-per-pole threshold 4a actually requires, which is why 4a's qualifying-donor counts (36 microglia, 78 astrocyte) come out much lower than these near-universal presence counts: most donors carry a handful of cells at a given pole, not enough to form a stable per-donor centroid.

### 2a — lineage reference (clean upper anchor): astrocyte vs microglia

Measure the cosine distance between astrocytes and microglia in the embedding, the largest clean biological contrast available. Two forms are reported, both dimensionless and directly comparable to detector #1's 0.0051: the **centroid** distance (`1 - cos` between the two lineage mean-vectors), and a spread-aware **median cross-lineage pairwise** distance (median of `1 - cos` over random astrocyte-microglia cell pairs). Centroid distance ignores within-lineage spread and so can overstate separability; pairwise captures individual-cell variation the way detector #1's per-cell median does, rather than only the group-mean shift, so it is read as the more representative of the two -- though neither is a strict match, since detector #1 measures the same cell's own displacement across model versions while both these forms measure separation between different cells. Placing 0.0051 against this anchor shows what fraction of the embedding's dominant biological axis the continued-pretraining drift represents -- expected to be a small fraction, since lineage is a far larger difference than continued pretraining could impose.

In [8]:
DETECTOR1_V2_TEST = 0.0051   # colab_11 v2 detector #1 held-out drift (v1 was 0.0057)
PAIR_SAMPLE = 3000           # cells sampled per group for the pairwise-distance estimate

def centroid_cos_dist(Xa, Xb):
    ca, cb = Xa.mean(0), Xb.mean(0)
    cos = float(ca @ cb) / (np.linalg.norm(ca) * np.linalg.norm(cb) + 1e-12)
    return 1.0 - cos

def median_pairwise_cos_dist(Xa, Xb, n=PAIR_SAMPLE, seed=0):
    rng = np.random.default_rng(seed)   # fresh per call -> reproducible regardless of cell run-order
    ia = rng.choice(len(Xa), size=min(n, len(Xa)), replace=False)
    ib = rng.choice(len(Xb), size=min(n, len(Xb)), replace=False)
    Ua = Xa[ia] / (np.linalg.norm(Xa[ia], axis=1, keepdims=True) + 1e-12)
    Ub = Xb[ib] / (np.linalg.norm(Xb[ib], axis=1, keepdims=True) + 1e-12)
    sims = Ua @ Ub.T          # cross pairs, cosine similarity
    return float(1.0 - np.median(sims))

Xastro = X[lineage == "astrocyte"]
Xmicro = X[lineage == "microglia"]

lin_centroid = centroid_cos_dist(Xastro, Xmicro)
lin_pairwise = median_pairwise_cos_dist(Xastro, Xmicro)
print(f"lineage (astro vs micro) | centroid cos-dist: {lin_centroid:.4f} "
      f"| median pairwise cos-dist: {lin_pairwise:.4f}")
print(f"detector #1 v2 drift {DETECTOR1_V2_TEST:.4f} = "
      f"{100*DETECTOR1_V2_TEST/lin_centroid:.1f}% of centroid, "
      f"{100*DETECTOR1_V2_TEST/lin_pairwise:.1f}% of pairwise lineage distance")

lineage (astro vs micro) | centroid cos-dist: 0.0503 | median pairwise cos-dist: 0.0982
detector #1 v2 drift 0.0051 = 10.1% of centroid, 5.2% of pairwise lineage distance


> **Interpretation — 2a.** Lineage centroid distance is 0.0503 and pairwise is 0.0982 -- pairwise larger than centroid, as expected, since it carries the within-lineage cell-to-cell spread that a pure mean-vector comparison discards. Detector #1's 0.0051 is 10.1% of centroid and 5.2% of pairwise lineage distance: a small fraction of the largest, cleanest biological axis in this embedding, exactly as predicted going in. This also serves as a sanity check on the measurement approach itself -- if 0.0051 had come out as a large fraction of the lineage anchor, that would have signaled something broken in the distance computation or units, and it does not. Note the caveat in the markdown above: strictly, detector #1 (a same-cell before/after displacement) is not the same kind of quantity as either form measured here (both are between-different-cells separations) -- these numbers place 0.0051 on a scale, not claim an exact equivalence.

### 3a — substate reference, pooled (donor/study-confounded): homeostatic vs activated, resting vs reactive

Compute the substate contrast within each lineage across all cells -- homeostatic vs activated microglia, resting vs reactive astrocytes -- using the same centroid and pairwise cosine-distance forms as the lineage anchor. This is a smaller, more continued-pretraining-relevant contrast than lineage. It is reported with an explicit warning: the substate labels are strongly donor/study-confounded (the activated and reactive buckets concentrate in a minority of donors), so each pole's centroid mixes real substate biology with donor/batch signal. That mixing is expected to inflate the pooled distance above the within-donor value, but is not guaranteed to -- if donor-level batch offsets are not aligned with the substate split, pooling can just as easily wash the signal out and deflate it instead. This number alone is not a trustworthy reference in either direction; 4a's within-donor value is the deconfounded reference, and the pooled-minus-within-donor gap measured there is what actually tells you which way (and how far) the pooled number missed.

In [9]:
pooled = {}
for lin, (p1, p2) in POLES.items():
    Xp1 = X[(lineage == lin) & (substate == p1)]
    Xp2 = X[(lineage == lin) & (substate == p2)]
    c = centroid_cos_dist(Xp1, Xp2)
    pw = median_pairwise_cos_dist(Xp1, Xp2)
    pooled[lin] = (c, pw)
    print(f"{lin:10s} {p1} vs {p2:12s} | centroid: {c:.4f} | pairwise: {pw:.4f} "
          f"| n={len(Xp1)}/{len(Xp2)}")
    print(f"           detector #1 v2 drift {DETECTOR1_V2_TEST:.4f} = "
          f"{100*DETECTOR1_V2_TEST/c:.1f}% of centroid, "
          f"{100*DETECTOR1_V2_TEST/pw:.1f}% of pairwise (POOLED -- confound-inflated)")

microglia  homeostatic vs activated    | centroid: 0.0100 | pairwise: 0.0627 | n=25845/12109
           detector #1 v2 drift 0.0051 = 51.1% of centroid, 8.1% of pairwise (POOLED -- confound-inflated)
astrocyte  resting vs reactive     | centroid: 0.0029 | pairwise: 0.0571 | n=48147/28465
           detector #1 v2 drift 0.0051 = 176.4% of centroid, 8.9% of pairwise (POOLED -- confound-inflated)


> **Interpretation — 3a.** Pooled substate distances: microglia centroid 0.0100 / pairwise 0.0627; astrocyte centroid 0.0029 / pairwise 0.0571. Microglia behaves as expected (pairwise > centroid, both non-trivial). Astrocyte is the striking case: its pooled centroid distance is nearly zero -- the resting and reactive population *mean* vectors, pooled across all 145 donors, sit almost on top of each other -- while its pooled pairwise distance (0.0571) is close to microglia's. That combination means individual resting vs. reactive astrocyte cells differ substantially, but the two poles' average positions barely differ once pooled across the full donor/study mix: a signature of donor/study scatter within each pole diluting the group-mean separation without shrinking the underlying cell-to-cell spread. This cell is explicitly a diagnostic, not a trusted reference -- the full read, including whether pooling here inflates or deflates relative to the deconfounded value, is in 4a.

### 4a — substate reference, within-donor (deconfounded, primary): the meaningful-change reference

Repeat the substate contrast, but compute it *inside each donor* and then take the median across donors, in both forms used elsewhere in this notebook: the centroid cosine distance between that donor's two substate poles, and (as in 2a) a spread-aware median pairwise cosine distance, sampled within the same donor's two pole groups. The median of each of these per-donor distances, across all qualifying donors, is the within-donor reference. Because each distance is measured between two cell groups from the same person, donor and (since donors belong to one study) study differences cannot enter it -- what remains is the biological substate difference at the scale continued pretraining plausibly operates on. Pairwise is read as the more representative of the two forms here, for the same reason as 2a; centroid is reported alongside it for continuity with 3a. This is the reference 0.0051 is primarily read against. The number of qualifying donors and the spread of per-donor distances are printed: if too few donors carry both poles, the median is unstable and the cell says so, in which case the pooled value stands in as a fallback -- 3a's own comparison to this cell determines whether that fallback reads as an upper or lower bound, since pooling is not guaranteed to inflate the substate distance (see 3a).

In [10]:
MIN_CELLS_PER_POLE = 50   # per-donor minimum at each pole to form a stable centroid
MIN_DONORS = 5            # fewer qualifying donors than this -> within-donor median is unstable
assert "pooled" in dir(), "run cell 3a before 4a -- 4a reports the pooled-vs-within-donor gap"

for lin, (p1, p2) in POLES.items():
    centroid_dists = []
    pairwise_dists = []
    for d in np.unique(donor):
        m1 = (donor == d) & (lineage == lin) & (substate == p1)
        m2 = (donor == d) & (lineage == lin) & (substate == p2)
        if m1.sum() >= MIN_CELLS_PER_POLE and m2.sum() >= MIN_CELLS_PER_POLE:
            centroid_dists.append(centroid_cos_dist(X[m1], X[m2]))
            pairwise_dists.append(median_pairwise_cos_dist(X[m1], X[m2]))
    centroid_dists = np.array(centroid_dists)
    pairwise_dists = np.array(pairwise_dists)
    n_don = len(centroid_dists)
    pooled_c, pooled_pw = pooled[lin]
    print(f"\n{lin}: {p1} vs {p2}")
    print(f"  qualifying donors (>= {MIN_CELLS_PER_POLE} cells each pole): {n_don}")
    if n_don < MIN_DONORS:
        print(f"  UNSTABLE -- fewer than {MIN_DONORS} donors carry both poles; "
              f"within-donor median not trustworthy. Pooled centroid {pooled_c:.4f} / pairwise {pooled_pw:.4f} "
              f"stand in as a fallback (see 3a for whether pooling inflates or deflates this lineage's substate distance).")
        continue
    wd_c_med = float(np.median(centroid_dists))
    c_q25, c_q75 = float(np.quantile(centroid_dists, 0.25)), float(np.quantile(centroid_dists, 0.75))
    wd_pw_med = float(np.median(pairwise_dists))
    pw_q25, pw_q75 = float(np.quantile(pairwise_dists, 0.25)), float(np.quantile(pairwise_dists, 0.75))
    print(f"  within-donor centroid cos-dist: median {wd_c_med:.4f} (IQR {c_q25:.4f}-{c_q75:.4f})")
    print(f"  pooled centroid cos-dist:        {pooled_c:.4f}  "
          f"-> pooled-minus-within-donor (centroid) = {pooled_c - wd_c_med:+.4f}")
    print(f"  within-donor pairwise cos-dist: median {wd_pw_med:.4f} (IQR {pw_q25:.4f}-{pw_q75:.4f})")
    print(f"  pooled pairwise cos-dist:        {pooled_pw:.4f}  "
          f"-> pooled-minus-within-donor (pairwise) = {pooled_pw - wd_pw_med:+.4f}")
    print(f"  detector #1 v2 drift {DETECTOR1_V2_TEST:.4f} = "
          f"{100*DETECTOR1_V2_TEST/wd_c_med:.1f}% of within-donor centroid, "
          f"{100*DETECTOR1_V2_TEST/wd_pw_med:.1f}% of within-donor pairwise "
          f"(pairwise is the more commensurable comparison -- detector #1 is itself a per-cell median)")


microglia: homeostatic vs activated
  qualifying donors (>= 50 cells each pole): 36
  within-donor centroid cos-dist: median 0.0044 (IQR 0.0027-0.0073)
  pooled centroid cos-dist:        0.0100  -> pooled-minus-within-donor (centroid) = +0.0056
  within-donor pairwise cos-dist: median 0.0365 (IQR 0.0309-0.0411)
  pooled pairwise cos-dist:        0.0627  -> pooled-minus-within-donor (pairwise) = +0.0262
  detector #1 v2 drift 0.0051 = 116.9% of within-donor centroid, 14.0% of within-donor pairwise (pairwise is the more commensurable comparison -- detector #1 is itself a per-cell median)

astrocyte: resting vs reactive
  qualifying donors (>= 50 cells each pole): 78
  within-donor centroid cos-dist: median 0.0108 (IQR 0.0066-0.0226)
  pooled centroid cos-dist:        0.0029  -> pooled-minus-within-donor (centroid) = -0.0080
  within-donor pairwise cos-dist: median 0.0442 (IQR 0.0380-0.0519)
  pooled pairwise cos-dist:        0.0571  -> pooled-minus-within-donor (pairwise) = +0.0128
  de

> **Interpretation — 4a (primary result).** Feasibility is resolved cleanly: 36 donors qualify for the microglia contrast and 78 for astrocyte, both far above the `MIN_DONORS=5` floor, so the within-donor median is stable for both lineages and neither fell back to the pooled value. (Not shown: what fraction of each pole's total cells these qualifying donors cover -- the notebook reports donor counts, not cell-coverage of the qualifying subset; a gap worth closing if this reference is revisited.)
>
> Centroid and pairwise diverge sharply within-donor too: microglia centroid 0.0044 vs. pairwise 0.0365 (8.3x), astrocyte centroid 0.0108 vs. pairwise 0.0442 (4.1x) -- consistent with 2a/3a, since pairwise always carries extra within-pole spread that centroid discards. The pooled-vs-within-donor gap also resolves cleanly under pairwise: pooling *inflates* the substate distance for both lineages (microglia +0.0262, astrocyte +0.0128), matching the donor/study-confound story in 3a's markdown. Under centroid, the same comparison flips sign for astrocyte (pooled 0.0029 < within-donor 0.0108, a -0.0080 "deflation") -- that flip is the mean-cancellation artifact identified in 3a's interpretation, specific to centroid distance, and it does not appear once within-pole spread is accounted for via pairwise. This is why pairwise, not centroid, is read as the primary number here.
>
> **Headline:** against the pairwise within-donor reference, detector #1's drift (0.0051) is **14.0%** of the real homeostatic-vs-activated microglia difference and **11.5%** of the real resting-vs-reactive astrocyte difference, measured inside the same donor so no donor/study confound can inflate or deflate it. (Centroid would put these at 116.9% / 47.0% -- much larger -- but as noted in 2a, pairwise is read as the more representative metric here, not a strictly commensurable one; neither number is an exact match to what detector #1 measures, so both are read as scale placements, not equivalences.)
>
> Placed alongside every other contrast measured in this notebook, drift sits in a consistent **5-14%** band that grows monotonically as the reference narrows from lineage (5.2%) to pooled substate (8.1-8.9%) to within-donor substate (11.5-14.0%) -- the pattern a well-calibrated measurement should show, with no inversions or outliers.
>
> **Scope limit:** this places 0.0051 as a real, modest -- not negligible, not dominant -- fraction of genuine biological cell-state difference. It establishes *magnitude* only. Whether that ~12% represents a coherent, biologically sensible shift or an equally-sized shift in a direction that means nothing is not addressed by this notebook; that is what evals #1-3 test.